In [7]:
%load_ext autoreload
%autoreload 2

import os
import sys
import matplotlib.pyplot as plt
from aicsimageio import AICSImage
import platform
from scipy.ndimage import gaussian_filter


sys.path.append(os.path.abspath(os.path.join(os.pardir, 'src')))
from data_processing import *

system = platform.system()

if system == 'Linux':
    home = '/home/gerard/data/confocal/'
elif system == 'Darwin':
    home = '/Users/gerard/data/confocal/'
elif system == "Windows":
    home = 'C:/Users/cviko/data/confocal/'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
date = '2026_06_05'
user = 'Carmina'

info = describe_acquisition( home + date + '_' + user + '/Project.lif')

....

══════════════════════════════════════════════════════════════
Scene 0: dec02_5x_brain1alphaprime_L
══════════════════════════════════════════════════════════════
  Objective  : HC PL APO CS2    63x/1.40 OIL
  NA / n     : 1.4 / 1.518
  Voxel size : XY = 0.0226 µm/px  |  Z = 0.1307 µm/step
  Shape      : 61 × 512 × 512  (Z × Y × X)

  ────────────────────────────────────── Acquisition sequences
  Seq 1  laser(s): 488 nm + 638 nm  [simultaneous]
         pinhole  : 20.00 µm  (0.209 AU)
         det ch1  501–610 nm  Leica/ALEXA 488         em = 519 nm
         det ch2  643–788 nm  Leica/Cy5               em = 670 nm
  Seq 2  laser(s): 552 nm
         pinhole  : 20.00 µm  (0.209 AU)
         det ch2  557–761 nm  Leica/ALEXA 546         em = 573 nm

  ────────────────────────────────────── Image channels (0-indexed)
  ch0  Leica/ALEXA 488         em=519 nm  σ_xy=3.45px (0.078µm) [✓]  σ_z=2.03px [✓]  → 3D
  ch1  Leica/Cy5               em=670 nm  σ_xy=4.45px (0.101µm) [✓]  σ_z=2.62px 

In [4]:
loader2(date,user,False, False)

/home/gerard/data/confocal/2026_06_05_Carmina
Number of series: 10
series 0: shape = (1, 3, 61, 512, 512)
series 1: shape = (1, 3, 134, 512, 512)
series 2: shape = (1, 3, 121, 512, 512)
series 3: shape = (1, 3, 88, 512, 512)
series 4: shape = (1, 3, 174, 512, 512)
series 5: shape = (1, 3, 121, 512, 512)
series 6: shape = (1, 3, 127, 512, 512)
series 7: shape = (1, 3, 197, 512, 512)
series 8: shape = (1, 3, 101, 512, 512)
series 9: shape = (1, 3, 94, 512, 512)


### DECONVOLUTION :

In [5]:


path_data = date + '_' + user + '/Project.lif'
path_data_all = home + path_data #os.path.join(hom

print(path_data_all)
img = AICSImage(path_data_all)
# print(len(img.scenes))
# print(img.scenes[0])



/home/gerard/data/confocal/2026_06_05_Carmina/Project.lif


In [25]:


scenes = list(range(len(info.keys())))
flagged2d= []
flagged3d = []
## DESIRED NUMBER OF ITERATIONS:
iterations = [
      #      1,        
            # 2,
            # #3,
            # 4,
             5,
            # # 6,
            # # 7, 
            # 10, 
            # 15,
            # 20,
            #20result[bg].std() / original[bg].std()
            ]

for scene in scenes:
    img.set_scene(img.scenes[scene])
    channels = list(range(len(info[list(info.keys())[scene]]['image_channels'])))
    for channel in channels:   
        stack = img.get_image_data("ZYX", T=0, C=channel)
        for num_iter in iterations:
            print(f'--------------> iteration {num_iter} for scene {scene} channel {channel}')
            result, sigma_xy_px = deconvolve(stack, path_data_all, channel=channel, scene=scene, num_iter=num_iter)
            
            flagged = analyze_deconvolution_results(result, stack)
            flagged3d.append(flagged)
            result, sigma_xy_px = deconvolve(stack, path_data_all, channel=channel, scene=scene, num_iter=num_iter, forced2d = True)
            flagged = analyze_deconvolution_results(result, stack)
            flagged2d.append(flagged)
            

--------------> iteration 5 for scene 0 channel 0
PSF (ch0): λ=519 nm | σ_xy=0.078 µm (3.45 px) | σ_z=0.265 µm (2.03 px) → 3D
2026_06_05_s0_ch0_deconv3d_iter_5.tif
0.115 of inspected frames flagged
PSF (ch0): λ=519 nm | σ_xy=0.078 µm (3.45 px) | σ_z=0.265 µm (2.03 px) → 2D-per-frame (σ_z=2.03 px < 2, Z undersampled)
2026_06_05_s0_ch0_deconv2d_iter_5.tif
0.131 of inspected frames flagged
--------------> iteration 5 for scene 0 channel 1
PSF (ch1): λ=670 nm | σ_xy=0.101 µm (4.45 px) | σ_z=0.342 µm (2.62 px) → 3D


KeyboardInterrupt: 

In [26]:
from joblib import Parallel, delayed


iterations = [
      #      1,        
           # 2,
            #3,
           # 4,
            5,
            # 6,
            # 7, 
            # 10, 
            # 15,
            # 20,
            #20result[bg].std() / original[bg].std()
            ]


# --- Phase 1: load all stacks sequentially (img is stateful) ---
stacks = {}
scene_channels = {}
for scene in scenes:
    img.set_scene(img.scenes[scene])
    channels = list(range(len(info[list(info.keys())[scene]]['image_channels'])))
    scene_channels[scene] = channels
    for channel in channels:
        stacks[(scene, channel)] = img.get_image_data("ZYX", T=0, C=channel)

# --- Phase 2: build flat job list ---
jobs = [
    (stacks[(scene, channel)], scene, channel, num_iter, forced2d)
    for scene in scenes
    for channel in scene_channels[scene]
    for num_iter in iterations
    for forced2d in [False, True]
]

def run_one(stack, scene, channel, num_iter, forced2d):
    result, sigma_xy_px = deconvolve(
        stack, path_data_all, channel=channel, scene=scene,
        num_iter=num_iter, forced2d=forced2d
    )
    flagged = analyze_deconvolution_results(result, stack)
    return (scene, channel, num_iter, forced2d, flagged)

# --- Run in parallel (threading shares imports & sys.path) ---
results = Parallel(n_jobs=8, backend='threading')(
    delayed(run_one)(*job) for job in jobs
)

# --- Reconstruct flagged lists ---
flagged3d = [r[-1] for r in results if not r[-2]]   # forced2d=False
flagged2d = [r[-1] for r in results if r[-2]]        # forced2d=True


PSF (ch0): λ=519 nm | σ_xy=0.078 µm (3.45 px) | σ_z=0.265 µm (2.03 px) → 2D-per-frame (σ_z=2.03 px < 2, Z undersampled)
PSF (ch1): λ=670 nm | σ_xy=0.101 µm (4.45 px) | σ_z=0.342 µm (2.62 px) → 2D-per-frame (σ_z=2.62 px < 2, Z undersampled)
PSF (ch1): λ=670 nm | σ_xy=0.101 µm (4.45 px) | σ_z=0.342 µm (2.62 px) → 3D
PSF (ch0): λ=519 nm | σ_xy=0.078 µm (3.45 px) | σ_z=0.265 µm (2.03 px) → 3D
PSF (ch2): λ=573 nm | σ_xy=0.086 µm (3.81 px) | σ_z=0.293 µm (2.24 px) → 3D
PSF (ch0): λ=519 nm | σ_xy=0.078 µm (3.45 px) | σ_z=0.265 µm (2.03 px) → 3D
PSF (ch2): λ=573 nm | σ_xy=0.086 µm (3.81 px) | σ_z=0.293 µm (2.24 px) → 2D-per-frame (σ_z=2.24 px < 2, Z undersampled)
PSF (ch0): λ=519 nm | σ_xy=0.078 µm (3.45 px) | σ_z=0.265 µm (2.03 px) → 2D-per-frame (σ_z=2.03 px < 2, Z undersampled)
2026_06_05_s0_ch1_deconv2d_iter_5.tif
0.000 of inspected frames flagged
PSF (ch1): λ=670 nm | σ_xy=0.101 µm (4.45 px) | σ_z=0.342 µm (2.62 px) → 3D
2026_06_05_s0_ch0_deconv2d_iter_5.tif
2026_06_05_s0_ch2_deconv2d_ite

KeyboardInterrupt: 

2026_06_05_s0_ch2_deconv3d_iter_5.tif


2026_06_05_s0_ch0_deconv3d_iter_5.tif
0.000 of inspected frames flagged
0.090 of inspected frames flagged
0.115 of inspected frames flagged
2026_06_05_s1_ch1_deconv2d_iter_5.tif
0.000 of inspected frames flagged
2026_06_05_s1_ch2_deconv2d_iter_5.tif
0.000 of inspected frames flagged
2026_06_05_s1_ch2_deconv3d_iter_5.tif
0.000 of inspected frames flagged
2026_06_05_s1_ch0_deconv3d_iter_5.tif
0.090 of inspected frames flagged
2026_06_05_s1_ch1_deconv3d_iter_5.tif
0.000 of inspected frames flagged


In [13]:
import os
os.cpu_count()

16

In [24]:
print(flagged3d[0:-1:3])
print(flagged3d[1:-1:3])
print(flagged3d[2:-1:3])


print(flagged2d[0:-1:3])
print(flagged2d[1:-1:3])
print(flagged2d[2:-1:3])





[0.13114754098360656, 0.11475409836065574, 0.0, 0.0, 0.0, 0.08955223880597014, 0.08208955223880597, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.6136363636363636, 0.6136363636363636, 0.0, 0.0, 0.0, 0.7816091954022989, 0.7816091954022989, 0.0, 0.0, 0.005747126436781609, 0.4628099173553719, 0.4628099173553719, 0.0, 0.0, 0.0, 0.6062992125984252, 0.5984251968503937, 0.0, 0.0, 0.0, 0.36548223350253806, 0.3604060913705584, 0.0, 0.0, 0.0, 0.49504950495049505, 0.49504950495049505, 0.0, 0.0, 0.019801980198019802, 0.6702127659574468, 0.6595744680851063, 0.0, 0.0, 0.0]
[0.11475409836065574, 0.11475409836065574, 0.0, 0.0, 0.0, 0.08955223880597014, 0.08208955223880597, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.6136363636363636, 0.6136363636363636, 0.0, 0.0, 0.0, 0.7816091954022989, 0.7758620689655172, 0.0, 0.005747126436781609, 0.005747126436781609, 0.4628099173553719, 0.4628099173553719, 0.0, 0.0, 0.0, 0.5984251968503937, 0.5984251968503937, 0.0, 0.0, 0.0, 0.36548223350253806, 0.3604060913705584, 0.0